In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Kaggle notebook code (GPU recommended)
# ------------------------------------

import json, os, random 
from collections import Counter, defaultdict

import torch
from torch.utils.data import DataLoader

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    # Trainer,
    DataCollatorWithPadding,
    set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = "/kaggle/input/data4good-updaedcontextsplit/train_80.json"  # provided path in this environment; on Kaggle set accordingly

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
CLASS_NAMES = ["factual", "contradiction", "irrelevant"]

def build_premise(ex):
    # You can tweak this, but keep it consistent across runs
    ctx = (ex.get("context") or "").strip()
    q   = (ex.get("question") or "").strip()
    if ctx:
        return f"{ctx}\n\nQuestion: {q}"
    return f"Question: {q}"

def build_hypothesis(ex):
    return (ex.get("answer") or "").strip()

def get_mnli_label_ids(model_config):
    """
    Returns dict: {"entailment": id, "contradiction": id, "neutral": id}
    Works across models where config.label2id may be inconsistent/cased.
    """
    l2i = model_config.label2id or {}
    # normalize
    norm = {str(k).lower(): int(v) for k, v in l2i.items()}
    out = {}

    # common names in MNLI configs
    for key in ["entailment", "contradiction", "neutral"]:
        if key in norm:
            out[key] = norm[key]

    # sometimes keys are like "LABEL_0"/"LABEL_1"/"LABEL_2"
    # and config.id2label provides meaning
    if len(out) < 3 and getattr(model_config, "id2label", None):
        i2l = {int(k): str(v).lower() for k, v in model_config.id2label.items()}
        for i, name in i2l.items():
            if "entail" in name:
                out["entailment"] = i
            elif "contra" in name:
                out["contradiction"] = i
            elif "neutral" in name:
                out["neutral"] = i

    # final fallback (MNLI default often: 0=contradiction,1=neutral,2=entailment)
    if len(out) < 3:
        out = {"contradiction": 0, "neutral": 1, "entailment": 2}

    return out

# dataset label mapping:
# factual -> entailment, contradiction -> contradiction, irrelevant -> neutral
def to_mnli_target_id(ex, mnli_ids):
    t = ex["type"]
    if t == "factual":
        return mnli_ids["entailment"]
    if t == "contradiction":
        return mnli_ids["contradiction"]
    if t == "irrelevant":
        return mnli_ids["neutral"]
    raise ValueError(f"Unknown type: {t}")

def preds_to_task_label(pred_mnli_id, mnli_ids):
    # reverse mapping MNLI decision -> task class
    if pred_mnli_id == mnli_ids["entailment"]:
        return "factual"
    if pred_mnli_id == mnli_ids["contradiction"]:
        return "contradiction"
    if pred_mnli_id == mnli_ids["neutral"]:
        return "irrelevant"
    # if model outputs something unexpected, call it irrelevant
    return "irrelevant"

def compute_metrics_task(y_true, y_pred, title=""):
    # overall
    overall_acc = float(np.mean([a == b for a, b in zip(y_true, y_pred)]))
    overall_f1_macro = float(f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro"))

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in CLASS_NAMES:
        idxs = [i for i, yt in enumerate(y_true) if yt == c]
        if len(idxs) == 0:
            per_class_acc[c] = None
        else:
            per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs]))

    report = classification_report(
        y_true, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0
    )

    # print clean summary
    print("\n" + "="*80)
    if title:
        print(title)
    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Overall F1 (macro): {overall_f1_macro:.4f}")
    print("\nPer-class Accuracy:")
    for c in CLASS_NAMES:
        v = per_class_acc[c]
        print(f"  {c:14s} {('NA' if v is None else f'{v:.4f}')}")
    print("\nPer-class F1:")
    for c in CLASS_NAMES:
        print(f"  {c:14s} {report[c]['f1-score']:.4f}")

    return {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1_macro,
        "per_class_accuracy": per_class_acc,
        "per_class_f1": {c: float(report[c]["f1-score"]) for c in CLASS_NAMES},
    }

In [ ]:
import os, json, gc, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
gc.collect(); torch.cuda.empty_cache()

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # use public model for reproducibility
# MODEL_NAME = "microsoft/deberta-v3-base-mnli"  # if you want faster + less OOM risk

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

MAX_LENGTH = 250
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

SPLIT_DIR = "./splits_80_10_10"     # where you saved train_80.json etc.
OUT_DIR = "./outputs_80_10_10"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# Load split json files
# -----------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json('/kaggle/input/data4good-updaedcontextsplit/train_80.json')
val_data   = load_json('/kaggle/input/data4good-updaedcontextsplit/val_10.json')
test_data  = load_json('/kaggle/input/data4good-updaedcontextsplit/test_10.json')

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist:", train_df["type"].value_counts().to_dict())
print("Val dist:", val_df["type"].value_counts().to_dict())
print("Test dist:", test_df["type"].value_counts().to_dict())

# -----------------------
# Tokenize
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Model
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# memory saver
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# -----------------------
# Metrics (paper required)
# -----------------------
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = metrics_from_preds(labels.tolist(), preds.tolist())
    # Trainer expects flat numeric values
    return {"accuracy": m["overall_accuracy"], "f1_macro": m["overall_f1_macro"]}

# -----------------------
# TrainingArguments (handles eval_strategy API mismatch)
# -----------------------
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

args_kwargs = dict(
    output_dir=os.path.join(OUT_DIR, "ft_model"),
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)
args_kwargs[eval_key] = "epoch"
args_kwargs = {k:v for k,v in args_kwargs.items() if k in allowed}

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_trainer,
)

gc.collect(); torch.cuda.empty_cache()
trainer.train()

# -----------------------
# Evaluate on VAL + TEST, save metrics + predictions
# -----------------------
def predict_and_save(split_name, df, ds):
    pred = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    conf = probs.max(axis=-1)
    true_ids = df["label"].values

    # metrics
    met = metrics_from_preds(true_ids.tolist(), pred_ids.tolist())
    met_row = {"model": MODEL_NAME, "split": split_name, **met}

    # predictions csv
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = [id2label[i] for i in pred_ids]
    out["pred_conf"] = conf
    for i,lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}.csv")
    out.to_csv(pred_path, index=False)

    return met_row, pred_path

val_metrics_row, val_pred_path = predict_and_save("val", val_df, val_ds)
test_metrics_row, test_pred_path = predict_and_save("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_metrics_row, test_metrics_row])
metrics_path = os.path.join(OUT_DIR, "finetune_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


Sizes: 16816 2102 2103
Train dist: {'factual': 13944, 'contradiction': 1454, 'irrelevant': 1418}
Val dist: {'factual': 1743, 'contradiction': 182, 'irrelevant': 177}
Test dist: {'factual': 1744, 'contradiction': 182, 'irrelevant': 177}


Map:   0%|          | 0/16816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2102 [00:00<?, ? examples/s]

Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


In [1]:
import os, gc, torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

gc.collect()
torch.cuda.empty_cache()

# Path where Trainer saved your best checkpoint
SAVED_DIR = os.path.join('/kaggle/working/outputs_80_10_10', "ft_model/checkpoint-1051")  # same as output_dir used in training

# Load tokenizer + fine-tuned model from disk
tokenizer = AutoTokenizer.from_pretrained(SAVED_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(SAVED_DIR)

# If you want to enforce label names (optional, but helps readability)
model.config.id2label = id2label
model.config.label2id = label2id

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

2026-01-29 14:43:22.517838: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769697802.768287      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769697802.842666      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769697803.477807      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697803.477853      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697803.477856      55 computation_placer.cc:177] computation placer alr

In [2]:

MAX_LENGTH = 250
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# Minimal args just for prediction (eval batch size matters)
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

pred_args_kwargs = dict(
    output_dir=os.path.join('/kaggle/working/outputs_80_10_10', "serapi_pred_only"),
    per_device_eval_batch_size=EVAL_BS,
    fp16=True,
    report_to="none",
)

pred_args_kwargs[eval_key] = "no"
pred_args_kwargs = {k:v for k,v in pred_args_kwargs.items() if k in allowed}
pred_args = TrainingArguments(**pred_args_kwargs)

pred_trainer = Trainer(
    model=model,
    args=pred_args,
    data_collator=data_collator,
)

LABELS = ["factual", "contradiction", "irrelevant"]



In [7]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json('/kaggle/input/data4good-updaedcontextsplit/train_80.json')
val_data   = load_json('/kaggle/input/data4good-updaedcontextsplit/val_10.json')
test_data  = load_json('/kaggle/input/data4good-updaedcontextsplit/test_10.json')

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

In [8]:


def metrics_table(y_true_str, y_pred_str):
    # overall
    overall_acc = float(np.mean(np.array(y_true_str) == np.array(y_pred_str)))
    overall_f1  = float(f1_score(y_true_str, y_pred_str, labels=LABELS, average="macro", zero_division=0))

    # per-class accuracy
    per_class_acc = {}
    for c in LABELS:
        idxs = np.where(np.array(y_true_str) == c)[0]
        per_class_acc[c] = float(np.mean(np.array(y_pred_str)[idxs] == c)) if len(idxs) else None

    # per-class f1
    rep = classification_report(
        y_true_str, y_pred_str, labels=LABELS, output_dict=True, zero_division=0
    )

    row = {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1,
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(rep["factual"]["f1-score"]),
        "f1_contradiction": float(rep["contradiction"]["f1-score"]),
        "f1_irrelevant": float(rep["irrelevant"]["f1-score"]),
    }
    return row

@torch.no_grad()
def predict_split(split_name, df, ds):
    pred = pred_trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    pred_labels = [id2label[int(i)] for i in pred_ids]

    true_labels = df["type"].astype(str).str.lower().tolist()

    # metrics
    met = metrics_table(true_labels, pred_labels)
    met_row = {"model": SAVED_DIR, "split": split_name, **met}

    # save predictions
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = pred_labels
    out["pred_conf"] = probs.max(axis=-1)
    for i, lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}_LOADED.csv")
    out.to_csv(pred_path, index=False)

    print(split_name, met)
    return met_row, pred_path

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

Map:   0%|          | 0/16816 [00:00<?, ? examples/s]

Map:   0%|          | 0/2102 [00:00<?, ? examples/s]

Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

In [ ]:
import os, json
from pathlib import Path
from collections import Counter
OUT_DIR = Path("./splits_80_10_10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Predict + metrics on VAL and TEST using LOADED model
val_row, val_pred_path = predict_split("val", val_df, val_ds)
test_row, test_pred_path = predict_split("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_row, test_row])
metrics_path = os.path.join('/kaggle/working/outputs_80_10_10', "serapi_added_finetune_metrics_LOADED.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [3]:
test_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_test_LOADED.csv")
val_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv")

In [10]:
from sklearn.metrics import accuracy_score, f1_score
LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

def metrics_by_initial_context(test_data, test_df_pred):
    assert len(test_data) == len(test_df_pred)

    results = {}

    # initially NO context (in original test_data)
    idx_no_ctx = test_data["context"].isna() | test_data["context"].eq("")
    df0 = test_df_pred.loc[idx_no_ctx]

    results["initially_no_context"] = metrics_from_preds(
        df0.label.tolist(),
        df0.pred_id.tolist()
    )
    print("initially_no_context:", df0.shape)

    # initially HAS context
    idx_ctx = ~idx_no_ctx
    df1 = test_df_pred.loc[idx_ctx]

    results["initially_has_context"] = metrics_from_preds(
        df1.label.tolist(),
        df1.pred_id.tolist()
    )
    print("initially_has_context:", df1.shape)

    return results


In [6]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

val_data   = load_json('/kaggle/input/data4good-original-data-split/val_10.json')
test_data  = load_json('/kaggle/input/data4good-original-data-split/test_10.json')

In [15]:
results = metrics_by_initial_context(pd.DataFrame(test_data), test_df_pred)

initially_no_context: (183, 13)
initially_has_context: (1920, 13)


In [16]:
pd.DataFrame([results['initially_no_context']])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.989071,0.978155,0.986667,1.0,1.0,0.993289,0.941176,1.0


In [17]:
pd.DataFrame([results['initially_has_context']])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.99375,0.98649,0.998118,0.951807,0.99375,0.996243,0.966361,0.996865


## Disagreements between GPT and deberta

In [13]:
import pandas as pd
import numpy as np
gpt_df = pd.read_csv("/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv")
test_df_pred = pd.read_csv("/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv")

In [5]:
test_df_pred

,answer,type,question,context,premise_text,hypothesis_text,label,pred_id,pred_label,pred_conf,prob_factual,prob_contradiction,prob_irrelevant
0,"In Popper's view, we don't need to search for ...",factual,What don't we need to look for about theories ...,"To Popper, who was an anti-justificationist, t...","To Popper, who was an anti-justificationist, t...","In Popper's view, we don't need to search for ...",0,0,factual,0.999918,0.999918,0.000078,0.000004
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,When was the phrase Irano-Aryan first used?,"The term ""Irano-Aryan"" (and its related forms ...","The term ""Irano-Aryan"" (and its related forms ...","The term ""Irano-Aryan"" was first used in 1878.",0,0,factual,0.985646,0.985646,0.014272,0.000082
2,The Augmented Foundation Programme is required...,factual,Where is the Augmented Programme required for ...,Fetuvalu offers the Cambridge syllabus. Motufo...,Fetuvalu offers the Cambridge syllabus. Motufo...,The Augmented Foundation Programme is required...,0,0,factual,0.999867,0.999867,0.000128,0.000005
3,The general election for Tucson's city council...,factual,When is Tucson's city council general election?,Both the council members and the mayor serve f...,Both the council members and the mayor serve f...,The general election for Tucson's city council...,0,0,factual,0.999894,0.999894,0.000100,0.000006
4,The organization in Plymouth named after Sir A...,factual,What Plymouth organization is named for Sir Al...,Plymouth is home to the Marine Biological Asso...,Plymouth is home to the Marine Biological Asso...,The organization in Plymouth named after Sir A...,0,0,factual,0.999905,0.999905,0.000091,0.000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,What is the head of each Municipio called?,"As of 2010[update], the city of Montevideo has...","As of 2010[update], the city of Montevideo has...",The head of each Municipio is called an alcald...,0,0,factual,0.999925,0.999925,0.000071,0.000004
2098,Alan Dershowitz characterized Sudan as a gover...,factual,How did Alan Dershozitz describe the Sudan?,Alan Dershowitz described Sudan as an example ...,Alan Dershowitz described Sudan as an example ...,Alan Dershowitz characterized Sudan as a gover...,0,0,factual,0.999941,0.999941,0.000053,0.000005
2099,The name of the tower that was the first purpo...,factual,What's the name of the wall tower that was the...,The name of the wall tower that was the first ...,The name of the wall tower that was the first ...,The name of the tower that was the first purpo...,0,0,factual,0.999096,0.999096,0.000889,0.000015
2100,"Thomas Young first used the term ""energy"" inst...",factual,"When did Thomas Young use the term ""energy"" in...","In 1807, Thomas Young was possibly the first t...","In 1807, Thomas Young was possibly the first t...","Thomas Young first used the term ""energy"" inst...",0,0,factual,0.999869,0.999869,0.000127,0.000004


In [3]:
test_df_pred[['answer', 'type', 'question', 'context']]

,answer,type,question,context
0,"In Popper's view, we don't need to search for ...",factual,What don't we need to look for about theories ...,"To Popper, who was an anti-justificationist, t..."
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,When was the phrase Irano-Aryan first used?,"The term ""Irano-Aryan"" (and its related forms ..."
2,The Augmented Foundation Programme is required...,factual,Where is the Augmented Programme required for ...,Fetuvalu offers the Cambridge syllabus. Motufo...
3,The general election for Tucson's city council...,factual,When is Tucson's city council general election?,Both the council members and the mayor serve f...
4,The organization in Plymouth named after Sir A...,factual,What Plymouth organization is named for Sir Al...,Plymouth is home to the Marine Biological Asso...
...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,What is the head of each Municipio called?,"As of 2010[update], the city of Montevideo has..."
2098,Alan Dershowitz characterized Sudan as a gover...,factual,How did Alan Dershozitz describe the Sudan?,Alan Dershowitz described Sudan as an example ...
2099,The name of the tower that was the first purpo...,factual,What's the name of the wall tower that was the...,The name of the wall tower that was the first ...
2100,"Thomas Young first used the term ""energy"" inst...",factual,"When did Thomas Young use the term ""energy"" in...","In 1807, Thomas Young was possibly the first t..."


In [4]:
gpt_df

,answer,type,context,question,model_label
0,"In Popper's view, we don't need to search for ...",factual,"To Popper, who was an anti-justificationist, t...",What don't we need to look for about theories ...,factual
1,"The term ""Irano-Aryan"" was first used in 1878.",factual,NaN,When was the phrase Irano-Aryan first used?,contradiction
2,The Augmented Foundation Programme is required...,factual,Fetuvalu offers the Cambridge syllabus. Motufo...,Where is the Augmented Programme required for ...,factual
3,The general election for Tucson's city council...,factual,Both the council members and the mayor serve f...,When is Tucson's city council general election?,factual
4,The organization in Plymouth named after Sir A...,factual,Plymouth is home to the Marine Biological Asso...,What Plymouth organization is named for Sir Al...,factual
...,...,...,...,...,...
2097,The head of each Municipio is called an alcald...,factual,"As of 2010[update], the city of Montevideo has...",What is the head of each Municipio called?,factual
2098,Alan Dershowitz characterized Sudan as a gover...,factual,Alan Dershowitz described Sudan as an example ...,How did Alan Dershozitz describe the Sudan?,factual
2099,The name of the tower that was the first purpo...,factual,NaN,What's the name of the wall tower that was the...,factual
2100,"Thomas Young first used the term ""energy"" inst...",factual,"In 1807, Thomas Young was possibly the first t...","When did Thomas Young use the term ""energy"" in...",factual


In [11]:
from sklearn.metrics import accuracy_score, classification_report
def metrics_from_preds(y_true, y_pred):
    # y_true = [id2label[i] for i in y_true_ids]
    # y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

In [14]:
pd.DataFrame([metrics_from_preds(gpt_df.type.tolist(), gpt_df.model_label.tolist())])

,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
0,0.972883,0.949824,0.971888,0.972527,0.983051,0.984598,0.876238,0.988636


In [15]:
import pandas as pd
import numpy as np

# -----------------------------
# 0) Paths (your uploaded files)
# -----------------------------
VAL_NLI_PATH  = "/kaggle/working/splits_80_10_10/predictions_val_LOADED.csv"
TEST_NLI_PATH = "/kaggle/working/splits_80_10_10/predictions_test_LOADED.csv"
VAL_GPT_PATH  = "/kaggle/input/gpt-labelled-val-dataset/val10-gpt-labelled.csv"
TEST_GPT_PATH = "/kaggle/input/gpt-labelled-dataset/test10-gpt-labelled.csv"

OUT_CSV  = "/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv"
OUT_TEX  = "/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.tex"

# -----------------------------
# 1) Helpers (robust to quirks)
# -----------------------------
VALID = {"factual", "contradiction", "irrelevant"}
NUM2LBL = {0: "factual", 1: "contradiction", 2: "irrelevant"}

def normalize_label(x):
    """Map labels to {factual, contradiction, irrelevant}; anything else -> NaN."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return NUM2LBL.get(int(x), np.nan)
    s = str(x).strip().lower()
    s = s.replace(" ", "_")
    return s if s in VALID else np.nan

def context_group(x):
    """Split by whether context is present vs absent."""
    if pd.isna(x):
        return "absent"
    if isinstance(x, str) and x.strip() == "":
        return "absent"
    # if your pipeline uses placeholders, add them here:
    if isinstance(x, str) and x.strip().lower() in {"none", "null", "na", "n/a", "no_context"}:
        return "absent"
    return "present"

def pair_key(a, b):
    """Unordered label-pair key for disagreements (Factual↔Contradiction etc.)."""
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return "↔".join(sorted([a, b]))

def merge_preds(nli_df, gpt_df):
    """
    Merge NLI preds with GPT preds using stable text keys.
    Assumes columns exist:
      - NLI: question, answer, type, context, pred_label (or pred_id/label variants)
      - GPT: question, answer, type, context, model_label
    """
    nli = nli_df.copy()
    gpt = gpt_df.copy()

    # Normalize labels
    # NLI file already has pred_label strings; fallback to pred_id if needed
    if "pred_label" in nli.columns:
        nli["mnli"] = nli["pred_label"].apply(normalize_label)
    elif "pred_id" in nli.columns:
        nli["mnli"] = nli["pred_id"].apply(normalize_label)
    else:
        raise ValueError("NLI file missing pred_label/pred_id.")

    if "model_label" not in gpt.columns:
        raise ValueError("GPT file missing model_label.")
    gpt["gpt"] = gpt["model_label"].apply(normalize_label)

    # Merge on the most stable shared keys
    key_cols = [c for c in ["question", "answer", "type", "context"] if c in nli.columns and c in gpt.columns]
    if len(key_cols) < 2:
        raise ValueError(f"Too few merge keys found. Found keys={key_cols}")

    merged = nli.merge(
        gpt[key_cols + ["gpt"]],
        on=key_cols,
        how="left",
        validate="one_to_one",
    )

    merged["context_split"] = merged["context"].apply(context_group)

    # Validity + disagreement flags
    merged["mnli_valid"] = merged["mnli"].isin(list(VALID))
    merged["gpt_valid"]  = merged["gpt"].isin(list(VALID))
    merged["both_valid"] = merged["mnli_valid"] & merged["gpt_valid"]

    merged["agree"]    = merged["both_valid"] & (merged["mnli"] == merged["gpt"])
    merged["disagree"] = merged["both_valid"] & (merged["mnli"] != merged["gpt"])

    merged["pair"] = np.where(
        merged["disagree"],
        merged.apply(lambda r: pair_key(r["mnli"], r["gpt"]), axis=1),
        np.nan
    )

    return merged

def disagreement_summary_table(merged, dataset_name):
    """
    One table with:
      - # examples
      - # comparable (both valid)
      - # GPT missing/invalid (captures "ERROR..." etc.)
      - # disagreements and % among comparable
      - label-pair breakdown counts and % among disagreements
    """
    rows = []
    for split, g in merged.groupby("context_split", dropna=False):
        n = len(g)
        comparable = int(g["both_valid"].sum())
        gpt_bad = int((~g["gpt_valid"]).sum())  # includes NaN + non-label strings
        dis = int(g["disagree"].sum())

        dis_rate = (dis / comparable * 100.0) if comparable > 0 else np.nan

        pair_counts = g.loc[g["disagree"], "pair"].value_counts()

        def pc(k): return int(pair_counts.get(k, 0))
        # NOTE: keys are sorted strings ("contradiction↔factual", etc.)
        fc = pc("contradiction↔factual")
        fi = pc("factual↔irrelevant")
        ci = pc("contradiction↔irrelevant")

        rows.append({
            "dataset": dataset_name,
            "context": split,
            "n": n,
            "comparable_(both_valid)": comparable,
            "gpt_missing_or_invalid": gpt_bad,
            "disagree_n": dis,
            "disagree_%": round(dis_rate, 3) if comparable > 0 else np.nan,

            "F↔C_n": fc,
            "F↔I_n": fi,
            "C↔I_n": ci,

            "F↔C_%_of_dis": round((fc / dis * 100.0), 1) if dis > 0 else np.nan,
            "F↔I_%_of_dis": round((fi / dis * 100.0), 1) if dis > 0 else np.nan,
            "C↔I_%_of_dis": round((ci / dis * 100.0), 1) if dis > 0 else np.nan,
        })

    out = pd.DataFrame(rows)

    # Drop splits with zero comparable examples (avoids junk rows when "absent" is tiny)
    out = out[out["comparable_(both_valid)"] > 0].reset_index(drop=True)
    return out

def one_sentence_story(tab):
    """
    Generates a single paper-friendly sentence from the table.
    (You can paste this sentence under the table.)
    """
    # Aggregate across contexts per dataset
    sents = []
    for ds, g in tab.groupby("dataset"):
        comparable = g["comparable_(both_valid)"].sum()
        dis = g["disagree_n"].sum()
        rate = (dis / comparable * 100.0) if comparable > 0 else np.nan

        fc = g["F↔C_n"].sum()
        fi = g["F↔I_n"].sum()
        ci = g["C↔I_n"].sum()

        top_pair = max([("Factual↔Contradiction", fc), ("Factual↔Irrelevant", fi), ("Contradiction↔Irrelevant", ci)], key=lambda x: x[1])
        top_share = (top_pair[1] / dis * 100.0) if dis > 0 else np.nan

        sents.append((ds, rate, top_pair[0], top_share))

    # Prefer val+test in one sentence if both exist
    if set(tab["dataset"]) >= {"val", "test"}:
        v = [x for x in sents if x[0] == "val"][0]
        t = [x for x in sents if x[0] == "test"][0]
        return (
            f"MNLI and GPT disagree on {v[1]:.2f}% (val) and {t[1]:.2f}% (test) of comparable examples, "
            f"and most disagreements are {t[2]} flips (≈{t[3]:.1f}% of test disagreements), "
            f"showing remaining uncertainty concentrates on polarity rather than relevance."
        )

    # Fallback single-dataset sentence
    ds, rate, pair, share = sents[0]
    return (
        f"MNLI and GPT disagree on {rate:.2f}% of comparable examples ({ds}), "
        f"dominated by {pair} flips (≈{share:.1f}% of disagreements), "
        f"indicating uncertainty concentrates on polarity rather than relevance."
    )








In [16]:
# -----------------------------
# 2) Load + merge (val/test)
# -----------------------------
val_nli  = pd.read_csv(VAL_NLI_PATH)
test_nli = pd.read_csv(TEST_NLI_PATH)
val_gpt  = pd.read_csv(VAL_GPT_PATH)
test_gpt = pd.read_csv(TEST_GPT_PATH)

val_merged  = merge_preds(val_nli,  val_gpt)
test_merged = merge_preds(test_nli, test_gpt)


In [17]:
# -----------------------------
# 3) Build the ONE table
# -----------------------------
tab_val  = disagreement_summary_table(val_merged,  "val")
tab_test = disagreement_summary_table(test_merged, "test")
tab = pd.concat([tab_val, tab_test], ignore_index=True)

# Save for paper pipeline
tab.to_csv(OUT_CSV, index=False)

# LaTeX (booktabs-friendly)
latex = tab.to_latex(index=False, escape=False)
with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex)

print("Wrote:", OUT_CSV)
print("Wrote:", OUT_TEX)

Wrote: /kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv
Wrote: /kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.tex


In [18]:
# -----------------------------
# 4) The ONE sentence to paste
# -----------------------------
sentence = one_sentence_story(tab)
print("\nSentence for paper:\n", sentence)


Sentence for paper:
 MNLI and GPT disagree on 1.46% (val) and 1.42% (test) of comparable examples, and most disagreements are Factual↔Contradiction flips (≈96.3% of test disagreements), showing remaining uncertainty concentrates on polarity rather than relevance.


In [19]:

# -----------------------------
# 5) (Optional) sanity checks
# -----------------------------
# How many GPT rows were unusable (e.g., 'ERROR OCCURRED PLEASE FIX')?
print("\nGPT unusable labels:")
print("val :", int((~val_merged['gpt_valid']).sum()))
print("test:", int((~test_merged['gpt_valid']).sum()))

# Quick peek at the most common disagreement pairs overall
pairs = pd.concat([val_merged.loc[val_merged["disagree"], "pair"],
                   test_merged.loc[test_merged["disagree"], "pair"]])
print("\nTop disagreement pairs overall:")
print(pairs.value_counts().head(10))


GPT unusable labels:
val : 187
test: 195

Top disagreement pairs overall:
pair
contradiction↔factual    50
factual↔irrelevant        5
Name: count, dtype: int64


In [20]:
pd.read_csv("/kaggle/working/splits_80_10_10/disagreement_mnli_vs_gpt_table.csv")

,dataset,context,n,comparable_(both_valid),gpt_missing_or_invalid,disagree_n,disagree_%,F↔C_n,F↔I_n,C↔I_n,F↔C_%_of_dis,F↔I_%_of_dis,C↔I_%_of_dis
0,val,present,2102,1915,187,28,1.462,24,4,0,85.7,14.3,0.0
1,test,present,2102,1908,194,27,1.415,26,1,0,96.3,3.7,0.0
